[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/radar/radar.ipynb)

# Radar Ranging

This example reconstructs a moving reflector's spacetime trajectory from the emission and arrival times of light pulses measured on a single inertial clock. Half the sum of each pulse's emission and reception readings assigns a radar time to the reflection event, half their difference assigns its distance in light-travel units, and the stretching of echo intervals reveals the reflector's relative velocity.

Events in projective spacetime are modelled as intrinsic homogeneous points (`Event-`), while straight worldlines and simultaneous clock slices are modelled as extrinsic planes (`Plane+`). Joining an emission event to an ideal null direction and intersecting the target's worldline via `(Event & outward) ^ reflector` forms a linear map (`Event- <- Event-`) on homogeneous events whose composition with the return leg maps emissions directly to receptions. Carrying events, worldlines, and the clock plane together under one Poincaré motor `motor >> clock` leaves every proper-time reading `clock & event` unchanged even as the line of simultaneity tilts across the diagram.

In Minkowskian coordinate and matrix notation, as used by Dolby and Gull in [*On Radar Time and the Twin 'Paradox'*](https://arxiv.org/abs/gr-qc/0104077) to extend Einstein's midpoint radar assignment to general observers, tracing a pulse from $(x_0, t_-)$ along $x - x_0 = t - t_-$ against the worldline $x = d + v t$ requires solving coupled inhomogeneous linear equations for $(x_{\mathrm{ref}}, t_{\mathrm{ref}})$ and a second system for $t_+$, while changing observer frame applies a $2\times 2$ Lorentz boost $\Lambda(\eta) = \begin{pmatrix}\cosh\eta & \sinh\eta \\ \sinh\eta & \cosh\eta\end{pmatrix}$ centered at $(0, t_c)$ plus separate shift vectors $(\Delta x, \Delta t)^\top$, transforming worldline normals by the inverse transpose $\Lambda(\eta)^{-\top}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.relativity.radar import render, scenarios
from examples.util.animation import save_figure, show_animation, show_figure, show_states
from examples.util.panels import image_figure

ga = Algebra("x+t-w0")
mv = NumpyContext(ga).multivector
# Events are intrinsic homogeneous points: a nonzero common scale leaves their location unchanged.
Event = ga.gatype.antivector()                             # Antivector-
# An extrinsic plane pairs to zero with events on a worldline or simultaneous slice.
Plane = ga.gatype.vector()                                 # Vector+
Scalar = ga.gatype.scalar()                                # Scalar+
EventMap = ga.gatype((Event, Event))                       # Event- <- Event-

# The signed blade complement gives an origin with homogeneous weight mv.w & origin == 1.
origin = mv.w.dual()                                       # Event-
# Weight-zero events are ideal points: spatial and temporal directions.
across = mv.x.dual()                                       # Event-
time = -mv.t.dual()                                        # Event-
# Joining these directions to the origin gives light rays whose line squares are zero.
# Every ideal point itself squares to zero, including across and time.
outgoing = time + across                                   # Event-
incoming = time - across                                   # Event-

## Light pulses and echoes

Each light leg joins an event to a fixed ideal null direction and meets a straight worldline plane, producing a linear map on homogeneous events once normalized on a unit-weight reference event. Composing the outbound reflection map with the return map sends each emission event directly to its reception event on the transmitter's worldline.

In [ ]:
distance, velocity = scenarios.distance, scenarios.velocity
n_pulses, pulse_spacing = scenarios.n_pulses, scenarios.pulse_spacing
end_margin = scenarios.end_margin
n_echo_frames = scenarios.n_echo_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def echo_maps(receiver: Plane, reflector: Plane, outward: Event, inward: Event,
              reference: Event) -> tuple[EventMap, EventMap]:
    """The reflection and round-trip echo maps (`Event- <- Event-`) follow straight timelike worldlines along fixed null directions."""
    # Join an event to the outgoing ideal direction, then intersect the reflector's worldline.
    reflection = (Event & outward) ^ reflector             # [] Event- <- Event-
    returning = (Event & inward) ^ receiver                # [] Event- <- Event-
    # Fixed directions and straight worldlines make output weight a constant times input weight.
    # A unit-weight reference fixes that one scale for the whole map, preserving its linearity.
    reflection = reflection / (mv.w & reflection(reference))  # [] Event- <- Event-
    returning = returning / (mv.w & returning(reference))     # [] Event- <- Event-
    echo = returning(reflection)                              # [] Event- <- Event-
    return reflection, echo


# xt squares to +1 and generates boosts; arctanh(velocity) is the reflector's rapidity.
# xw squares to zero and generates the spatial translation, applied after the boost.
placement = (mv.xw * (distance / 2)).exp() * (mv.xt * (np.arctanh(velocity) / 2)).exp()  # [] Motor+
target = placement >> mv.x                                                               # [] Plane+
emitted = origin + time * (np.arange(n_pulses) * pulse_spacing)                          # [n_pulses] Event-
reflection, echo = echo_maps(mv.x, target, outgoing, incoming, origin)
reflected, received = reflection(emitted), echo(emitted)                                 # [n_pulses] Event-, Event-

# Finite worldline segments bound the drawing after the last reception.
finish = (mv.t & received[-1]) + end_margin                                              # [] Scalar+
target_end = target ^ (mv.t - mv.w * finish)                                             # [] Event-
target_end = target_end / (mv.w & target_end)                                            # [] Event-
target_start = placement >> origin                                                       # [] Event-
worldlines = stack((stack((origin, origin + time * finish)),
                    stack((target_start, target_end))))                                  # [2, 2] Event-

show_animation(render.exchange_animation(worldlines, emitted, reflected, received, n_echo_frames),
               "radar_echoes", frame_duration_ms, n_frames=n_echo_frames);

Coordinate time runs upward at unit light speed so null rays travel along 45-degree diagonals. Dark ink draws the transmitting clock's worldline with emission and reception readings stamped beside each event, grey draws the receding reflector's worldline, orange traces the outbound pulses to their reflections, and blue traces the returning echoes stretched further apart than the emissions.

## Recovering the reflector trajectory

Pairing the clock plane with the sum and difference of reception and emission events assigns each echo its midpoint radar time and half-interval range, locating the reflection in spacetime without a clock on the target. Comparing the arrival-to-emission interval stretch across pulses recovers the relative Doppler factor, velocity, and initial range.

In [ ]:
def radar_assignment(emitted: Event, received: Event, clock: Plane, reference: Event,
                     tangent: Event, spatial: Event) -> tuple[Event, Event, Scalar, Scalar]:
    """Einstein's midpoint convention assigns each reflection its event, simultaneous clock event, radar time, and radar range (`[...] Event-, Event-, Scalar+, Scalar+`)."""
    radar_time = (clock & (received + emitted)) / 2        # [...] Scalar+
    radar_range = (clock & (received - emitted)) / 2       # [...] Scalar+
    simultaneous = reference + tangent * radar_time        # [...] Event-
    located = simultaneous + spatial * radar_range         # [...] Event-
    return located, simultaneous, radar_time, radar_range


def reflector_motion(emitted: Event, received: Event, clock: Plane) -> tuple[Scalar, Scalar, Scalar]:
    """Two or more echoes on an inertial clock determine the echo stretch, relative velocity, and range at clock zero (`[] Scalar+, Scalar+, Scalar+`)."""
    # An inertial reflector makes the arrival time an affine function of emission time.
    stretch = (clock & (received[-1] - received[0])) / (clock & (emitted[-1] - emitted[0]))  # [] Scalar+
    # Radar range changes with the difference of the two intervals; radar time with their sum.
    velocity = (stretch - 1) / (stretch + 1)               # [] Scalar+
    first_time = (clock & (received[0] + emitted[0])) / 2  # [] Scalar+
    first_range = (clock & (received[0] - emitted[0])) / 2  # [] Scalar+
    initial_range = first_range - velocity * first_time    # [] Scalar+
    return stretch, velocity, initial_range


located, simultaneous, radar_time, radar_range = radar_assignment(
    emitted, received, mv.t, origin, time, across)
stretch, inferred_velocity, initial_range = reflector_motion(emitted, received, mv.t)

figure = image_figure(render.reconstruction_image(
    worldlines, emitted, reflected, received, located, simultaneous, radar_time, radar_range, mv.t,
    render.exchange_window))
save_figure(figure, "radar_reconstruction")
show_figure(figure);

Green dots mark all reconstructed reflection events along the grey target worldline, connected by a green segment. For the highlighted middle pulse, the shaded causal triangle joins emission `τ−` and reception `τ+` on the clock to the reflection, while the dashed green segment measures the radar range `R` out from the simultaneous clock event `T`.

In [ ]:
print("sent:", (mv.t & emitted).to_array())
print("returned:", (mv.t & received).to_array())
print("radar time:", radar_time.to_array())
print("range:", radar_range.to_array())
print("echo stretch:", stretch.to_array())
print("relative velocity (c):", inferred_velocity.to_array())
print("range at clock zero:", initial_range.to_array())

## Changing spacetime frame

Applying a common Lorentz boost and spacetime translation to the entire experiment carries the events, worldlines, and transmitting clock together. Because `motor >> clock` and `motor >> event` preserve their regressive pairing `clock & event`, the reconstructed radar times, ranges, and relative velocity remain invariant while the observer's line of simultaneity tilts across the background grid.

In [ ]:
final_rapidity = scenarios.final_rapidity
spatial_shift = scenarios.spatial_shift
time_shift = scenarios.time_shift
n_boost_frames = scenarios.n_boost_frames

In [ ]:
def boosted_frames(worldlines: Event, emitted: Event, reflected: Event, received: Event,
                   final_rapidity: float, spatial_shift: float, time_shift: float,
                   n_frames: int) -> Iterator[tuple[Event, Event, Event, Event, Event, Event, Scalar, Scalar, Plane]]:
    """Yields the worldlines, pulse events, radar reconstruction, and clock plane across `n_frames` Poincare transformations."""
    midpoint_time = (mv.t & (emitted[0] + received[-1])) / 2              # [] Scalar+
    centre = (-mv.tw * (midpoint_time / 2)).exp()                         # [] Motor+
    phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    fractions = (1 - np.cos(phases)) / 2                                  # [n_frames]
    for fraction in fractions:
        boost = (mv.xt * (final_rapidity * fraction / 2)).exp()           # [] Motor+
        # The translation generator squares to zero, so its exponential is one plus the generator.
        translation = ((mv.xw * spatial_shift - mv.tw * time_shift) * (fraction / 2)).exp()  # [] Motor+
        # Conjugating the boost by centre anchors it at the midpoint event.
        motor = translation * centre * boost * ~centre                    # [] Motor+
        # The clock's unit and zero reading move with the experiment.
        clock = motor >> mv.t                                             # [] Plane+
        sent, returned = motor >> emitted, motor >> received              # [...] Event-, Event-
        hit = motor >> reflected                                          # [...] Event-
        moved_worldlines = motor >> worldlines                            # [2, 2] Event-
        moved_origin = motor >> origin                                    # [] Event-
        moved_time, moved_across = motor >> time, motor >> across         # [] Event-, Event-
        located, simultaneous, radar_time, radar_range = radar_assignment(
            sent, returned, clock, moved_origin, moved_time, moved_across)
        yield moved_worldlines, sent, hit, returned, located, simultaneous, radar_time, radar_range, clock


streamed_frames = show_states(
    boosted_frames(worldlines, emitted, reflected, received, final_rapidity, spatial_shift, time_shift, n_boost_frames),
    render.frames_animation, name="radar_frames", duration_ms=frame_duration_ms, n_frames=n_boost_frames);

As the Poincaré transformation shears the worldlines across the fixed `(x, t)` axes while preserving the 45-degree slopes of the orange and blue light rays, the dashed green simultaneity segment tilts with the boosted clock so that `τ−`, `τ+`, `T`, and `R` keep their exact values.

In [ ]:
# checks
# Outbound and return light lines are null, located events match actual reflections, and radar invariants hold across frames.
assert (((emitted & reflected).squared()).abs() < 1e-10).all()
assert (((reflected & received).squared()).abs() < 1e-10).all()
assert ((mv.x & (located - reflected)).abs() < 1e-10).all()
assert ((mv.t & (located - reflected)).abs() < 1e-10).all()
assert (inferred_velocity - velocity).abs() < 1e-10
assert (initial_range - distance).abs() < 1e-10
peak_state = streamed_frames[n_boost_frames // 2]
assert ((peak_state[6] - radar_time).abs() < 1e-10).all()
assert ((peak_state[7] - radar_range).abs() < 1e-10).all()